In [ ]:
import logging
from logging.handlers import RotatingFileHandler

logging.basicConfig(
    filename='/home/rohan/projects/airflow/logs/testing.log',
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    force=True   # 🔑 important
)
handler = RotatingFileHandler('test.log',
                                maxBytes=1000000,
                                backupCount=3)
    
logging.debug("Detailed info (dev only)")
logging.info("General info")
logging.warning("Something unexpected")
logging.error("Error occurred")
logging.critical("System failure")

In [ ]:
import time

for attempt in range(3):
    try:
        raise Exception("API failed")
    except Exception as e:
        logging.warning(f"Attempt {attempt+1} failed: {e}")
        time.sleep(1)

In [ ]:
try:
    1 / 0
except Exception:
    logging.exception("Something failed")

In [ ]:
try:
    result = 10 / 0
except Exception as e:
    logging.error(f"Error occurred: {e}")

In [1]:
from pyspark.sql import SparkSession
import logging


spark = SparkSession.builder \
    .appName("iceberg-local") \
    .config("spark.jars",
            "/home/rohan/projects/airflow/data_warehouse/iceberg-spark-runtime-4.0_2.13-1.10.1.jar") \
    .config("spark.sql.extensions",
            "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.local",
            "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.local.type", "hadoop") \
    .config("spark.sql.catalog.local.warehouse",
            "/home/rohan/projects/airflow/data_warehouse/iceberg_warehouse") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("Iceberg ready:", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/28 17:14:18 WARN Utils: Your hostname, Rohan, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/03/28 17:14:18 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
26/03/28 17:14:18 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


Iceberg ready: 4.1.1


In [2]:
spark.sql("SHOW CATALOGS").show()
spark.sql("SHOW NAMESPACES IN local").show()

+-------------+
|      catalog|
+-------------+
|spark_catalog|
+-------------+

+---------+
|namespace|
+---------+
|       db|
+---------+



In [3]:
spark.sql("SHOW TABLES IN local.db").show()

+---------+--------------------+-----------+
|namespace|           tableName|isTemporary|
+---------+--------------------+-----------+
|       db|         order_items|      false|
|       db|product_category_...|      false|
|       db|            products|      false|
|       db|      order_payments|      false|
|       db|  gold_order_summary|      false|
|       db|     silver_products|      false|
|       db|           customers|      false|
|       db|              orders|      false|
|       db|gold_state_paymen...|      false|
|       db|silver_order_reviews|      false|
|       db|         geolocation|      false|
|       db|silver_order_paym...|      false|
|       db|gold_revenue_by_o...|      false|
|       db|             sellers|      false|
|       db|  silver_order_items|      false|
|       db|       order_reviews|      false|
|       db|    silver_customers|      false|
|       db|       silver_orders|      false|
+---------+--------------------+-----------+



In [14]:
df = spark.read.format("iceberg").load("local.db.silver_products")
df.show(3)

+---------------------+--------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+-----------------------------+
|product_category_name|          product_id|product_name_lenght|product_description_lenght|product_photos_qty|product_weight_g|product_length_cm|product_height_cm|product_width_cm|product_category_name_english|
+---------------------+--------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+-----------------------------+
|          eletronicos|750cf819d12719192...|                 31|                       806|                 1|             263|               18|               12|              16|                  electronics|
|          eletronicos|75b433ca888fe027b...|                 37|                       374|                 1|            1350|               33|           

In [ ]:
history = spark.sql("SELECT * FROM local.db.silver_products.history")
history.show(truncate=False)

+-----------------------+-------------------+---------+-------------------+
|made_current_at        |snapshot_id        |parent_id|is_current_ancestor|
+-----------------------+-------------------+---------+-------------------+
|2026-03-28 17:12:32.222|2979915690379119161|NULL     |true               |
+-----------------------+-------------------+---------+-------------------+



In [18]:
snapshots = spark.sql("SELECT * FROM local.db.silver_products.snapshots")
snapshots.show(truncate=False)

+-----------------------+-------------------+---------+---------+--------------------------------------------------------------------------------------------------------------------------------------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|committed_at           |snapshot_id        |parent_id|operation|manifest_list                                                                                                                                                 |summary                                                                       

In [8]:
df.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)



In [ ]:
# #    Query using snapshot ID
# spark.sql("SELECT * FROM local.db.sales VERSION AS OF 6926624095525269367;").show(truncate=False)
# spark.sql("select * from local.db.sales.manifests ").show(truncate=False)
# spark.sql("select * from local.db.sales.files ").show(truncate=False)
# # -- Partition info 
# spark.sql("SELECT * FROM local.db.sales.partitions").show(truncate=False)
# # -- Table schema (Spark way) 
# spark.sql("DESCRIBE TABLE local.db.sales").show(truncate=False)
# # -- Detailed table info 
# spark.sql("DESCRIBE EXTENDED local.db.sales").show(truncate=False)


+---------------------------+-------------------+------------------+----------------+-----------------+
|geolocation_zip_code_prefix|    geolocation_lat|   geolocation_lng|geolocation_city|geolocation_state|
+---------------------------+-------------------+------------------+----------------+-----------------+
|                       1037| -23.54562128115268|-46.63929204800168|       sao paulo|               SP|
|                       1046|-23.546081127035535|-46.64482029837157|       sao paulo|               SP|
|                       1046| -23.54612896641469|-46.64295148361138|       sao paulo|               SP|
|                       1041|  -23.5443921648681|-46.63949930627844|       sao paulo|               SP|
|                       1035|-23.541577961711493|-46.64160722329613|       sao paulo|               SP|
+---------------------------+-------------------+------------------+----------------+-----------------+
only showing top 5 rows


In [ ]:
# tables = spark.sql("SHOW TABLES IN local.db").collect()

# for row in tables:
#     table_name = row['tableName']
#     spark.sql(f"DROP TABLE local.db.{table_name}")

In [ ]:
# spark.sql("DROP NAMESPACE IF EXISTS local.db CASCADE")
# logging.info("Namespace dropped successfully.")